# 基于 Spark 的 B站"每周必看"视频数据分析

**班级**：23数据科学与大数据技术  
**学号**：XXXXXXXXXX  
**姓名**：XXX  

---

## 一、实验环境

| 项目 | 说明 |
|------|------|
| 操作系统 | Windows 10/11 |
| JDK 版本 | JDK 17 |
| Spark 版本 | 3.5.8 |
| Python 版本 | 3.11+ |
| 开发工具 | PyCharm / Jupyter Notebook |
| 可视化库 | matplotlib、seaborn |

## 二、数据集介绍

### 2.1 数据来源与背景

B站（bilibili）是国内知名的视频分享平台，其"每周必看"栏目每周精选优质视频推荐给用户体验。本数据集通过 B站官方 API 爬取了从 2019 年第 1 期至 2026 年第 22 期共 **377 期**的"每周必看"数据，涵盖约 **14000 条**视频记录。

### 2.2 数据格式

数据以 JSON 格式存储，每个文件对应一期"每周必看"，文件命名为 `week_001.json` ~ `week_377.json`。

### 2.3 主要字段说明

| 字段 | 类型 | 说明 |
|------|------|------|
| number | int | 期数编号 |
| config.subject | string | 本期主题 |
| config.name | string | 期数名称（含日期范围） |
| config.stime / etime | long | 起止时间戳 |
| aid | long | 视频 AV 号 |
| bvid | string | 视频 BV 号 |
| title | string | 视频标题 |
| tid / tname | int / string | 分区 ID / 分区名称 |
| pubdate | long | 投稿时间戳 |
| duration | int | 视频时长（秒） |
| desc | string | 视频简介 |
| rcmd_reason | string | 推荐理由 |
| owner.mid / owner.name | long / string | UP主 ID / 昵称 |
| stat.view | long | 播放量 |
| stat.danmaku | long | 弹幕数 |
| stat.reply | long | 评论数 |
| stat.favorite | long | 收藏数 |
| stat.coin | long | 投币数 |
| stat.share | long | 分享数 |
| stat.like | long | 点赞数 |

## 三、数据预处理

### 3.1 初始化 Spark 环境

本项目使用 **Spark Connect** 模式连接 Spark 3.5.8。Spark Connect 是 Spark 3.4+ 引入的客户端-服务器架构，客户端通过 gRPC 与 Spark Server 通信，无需在本地嵌入 JVM。

**启动 Spark Connect Server**（在运行 notebook 前，需在终端执行）：

```bash
# Linux / macOS
$SPARK_HOME/sbin/start-connect-server.sh

# Windows（手动启动）
spark-class org.apache.spark.sql.connect.SparkConnectServer
```

Server 默认监听 `sc://localhost:15002`。

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
import warnings
warnings.filterwarnings('ignore')

# ============================================================
# 通过 Spark Connect 连接 Spark 3.5.8 Server
# remote(): 指定 Spark Connect Server 的地址
# 格式: sc://<host>:<port>
# 前提: 需要先启动 Spark Connect Server
# ============================================================
spark = SparkSession.builder \
    .appName('Bilibili Weekly Picks Analysis') \
    .remote('sc://192.168.212.134:15002') \
    .getOrCreate()

print(f'Spark 版本: {spark.version}')
print(f'连接模式: Spark Connect (sc://192.168.212.134:15002)')
print('SparkSession 创建成功！')

Spark 版本: 3.5.8
连接模式: Spark Connect (sc://192.168.212.134:15002)
SparkSession 创建成功！


### 3.2 读取 JSON 数据

使用 Spark 的 `spark.read.json()` 批量读取 `data/raw/week_*.json` 文件。

In [6]:
# ============================================================
# 读取所有 JSON 文件
# 数据已上传至 HDFS，使用 HDFS 路径读取
# ============================================================
DATA_PATH = 'hdfs://localhost:9000/user/hadoop/bilibili/raw/week_*.json'

raw_df = spark.read.json(DATA_PATH)

print(f'原始数据总行数: {raw_df.count()}')
print(f'\n顶层字段:')
raw_df.printSchema()

原始数据总行数: 1059392


AnalysisException: Since Spark 2.3, the queries from raw JSON/CSV files are disallowed when the
referenced columns only include the internal corrupt record column
(named _corrupt_record by default). For example:
spark.read.schema(schema).csv(file).filter($"_corrupt_record".isNotNull).count()
and spark.read.schema(schema).csv(file).select("_corrupt_record").show().
Instead, you can cache or save the parsed results and then send the same query.
For example, val df = spark.read.schema(schema).csv(file).cache() and then
df.filter($"_corrupt_record".isNotNull).count().

### 3.3 展开嵌套结构

JSON 数据中 `videos` 字段是一个数组，需要用 `explode` 展开为多行，再提取各个子字段。

In [3]:
# ============================================================
# Step 1: explode videos 数组 → 每个视频一行
# Step 2: 提取嵌套字段，拍平为 DataFrame 列
# ============================================================

# 展开 videos 数组
exploded_df = raw_df.select(
    F.col('number').alias('week_number'),          # 期数
    F.col('config.name').alias('week_name'),        # 期数名称
    F.col('config.subject').alias('week_subject'),  # 本期主题
    F.col('config.stime').alias('week_start'),      # 起始时间戳
    F.explode('videos').alias('video')              # 展开视频数组
)

# 从展开后的 video 结构体中提取各字段
videos_df = exploded_df.select(
    'week_number', 'week_name', 'week_subject',
    # --- 视频基本信息 ---
    F.col('video.aid').alias('aid'),
    F.col('video.bvid').alias('bvid'),
    F.col('video.title').alias('title'),
    F.col('video.tid').alias('tid'),
    F.col('video.tname').alias('category'),         # 分区名称
    F.col('video.pid_name_v2').alias('sub_category'),  # 二级分区
    F.col('video.pubdate').alias('pubdate'),         # 投稿时间戳
    F.col('video.duration').alias('duration'),       # 时长（秒）
    F.col('video.desc').alias('description'),        # 简介
    F.col('video.rcmd_reason').alias('rcmd_reason'), # 推荐理由
    # --- UP主信息 ---
    F.col('video.owner.mid').alias('creator_mid'),
    F.col('video.owner.name').alias('creator_name'),
    # --- 互动数据 ---
    F.col('video.stat.view').alias('view'),
    F.col('video.stat.danmaku').alias('danmaku'),
    F.col('video.stat.reply').alias('reply'),
    F.col('video.stat.favorite').alias('favorite'),
    F.col('video.stat.coin').alias('coin'),
    F.col('video.stat.share').alias('share'),
    F.col('video.stat.like').alias('like'),
)

total_videos = videos_df.count()
print(f'展开后视频总条数: {total_videos}')
print(f'\n前 5 条数据预览:')
videos_df.show(5, truncate=40)

AnalysisException: [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column or function parameter with name `number` cannot be resolved. Did you mean one of the following? [`_corrupt_record`].;
'Aggregate [unresolvedalias(count(1), None)]
+- 'Project ['week_number, 'week_name, 'week_subject, 'video.aid AS aid#94, 'video.bvid AS bvid#95, 'video.title AS title#96, 'video.tid AS tid#97, 'video.tname AS category#98, 'video.pid_name_v2 AS sub_category#99, 'video.pubdate AS pubdate#100, 'video.duration AS duration#101, 'video.desc AS description#102, 'video.rcmd_reason AS rcmd_reason#103, 'video.owner.mid AS creator_mid#104, 'video.owner.name AS creator_name#105, 'video.stat.view AS view#106, 'video.stat.danmaku AS danmaku#107, 'video.stat.reply AS reply#108, 'video.stat.favorite AS favorite#109, 'video.stat.coin AS coin#110, 'video.stat.share AS share#111, 'video.stat.like AS like#112]
   +- 'Project ['number AS week_number#89, 'config.name AS week_name#90, 'config.subject AS week_subject#91, 'config.s

### 3.4 数据类型转换与时间处理

将 Unix 时间戳转换为日期格式，将时长从秒转换为分钟，便于后续分析。

In [ ]:
# ============================================================
# 时间戳 → 日期，秒 → 分钟
# from_unixtime: 将 Unix 时间戳转为日期字符串
# ============================================================
videos_df = videos_df.withColumn(
    'pub_date',                             # 投稿日期
    F.to_date(F.from_unixtime('pubdate'))
).withColumn(
    'pub_year',                             # 投稿年份
    F.year(F.col('pub_date'))
).withColumn(
    'pub_month',                            # 投稿月份
    F.month(F.col('pub_date'))
).withColumn(
    'duration_min',                         # 时长（分钟）
    F.round(F.col('duration') / 60.0, 1)
)

print('时间字段转换完成，新增字段: pub_date, pub_year, pub_month, duration_min')
videos_df.select('title', 'pub_date', 'pub_year', 'duration_min').show(5, truncate=40)

### 3.5 数据清洗

检查并处理缺失值、重复数据和异常值。

In [ ]:
# ============================================================
# 1. 检查关键字段的空值情况
# ============================================================
key_cols = ['aid', 'title', 'category', 'view', 'like', 'creator_name', 'duration']

print('=== 关键字段空值统计 ===')
for col in key_cols:
    null_count = videos_df.filter(F.col(col).isNull()).count()
    print(f'  {col:>15s}: {null_count} 条空值 ({null_count/total_videos*100:.2f}%)')

# ============================================================
# 2. 去除 aid 为空的记录（无法标识视频）
# ============================================================
clean_df = videos_df.filter(F.col('aid').isNotNull())

# ============================================================
# 3. 去重：同一个 aid 可能在不同期被重复推荐
#    保留第一次出现的那条（按 week_number 升序）
# ============================================================
from pyspark.sql.window import Window

dedup_window = Window.partitionBy('aid').orderBy('week_number')
clean_df = clean_df.withColumn('rn', F.row_number().over(dedup_window)) \
                   .filter(F.col('rn') == 1) \
                   .drop('rn')

clean_count = clean_df.count()
print(f'\n清洗后视频总数: {clean_count}')
print(f'去重移除: {total_videos - clean_count} 条')

In [ ]:
# ============================================================
# 4. 异常值检测：播放量为 0 或负数的记录
# ============================================================
abnormal = clean_df.filter(F.col('view') <= 0).count()
print(f'播放量 <= 0 的异常记录: {abnormal} 条')

# 过滤掉播放量为 0 的记录
clean_df = clean_df.filter(F.col('view') > 0)
print(f'最终有效视频数: {clean_df.count()} 条')

# ============================================================
# 5. 基本统计描述
# ============================================================
print('\n=== 数值字段统计摘要 ===')
clean_df.select('view', 'like', 'coin', 'favorite', 'share', 'danmaku', 'reply', 'duration_min') \
    .describe().show()

## 四、数据分析

### 4.1 分区（Category）分布分析

统计各分区的视频数量，了解"每周必看"中哪些类型的内容最受欢迎。

In [ ]:
# ============================================================
# 按分区分组，统计视频数量和平均播放量
# groupBy + agg: Spark 的核心聚合操作
# ============================================================
category_stats = clean_df.groupBy('category').agg(
    F.count('*').alias('video_count'),                        # 视频数量
    F.round(F.avg('view'), 0).alias('avg_view'),             # 平均播放量
    F.round(F.avg('like'), 0).alias('avg_like'),             # 平均点赞数
    F.sum('view').alias('total_view'),                       # 总播放量
).orderBy(F.desc('video_count'))

print('=== Top 15 分区（按视频数量排序）===')
category_stats.show(15, truncate=False)

### 4.2 播放量 Top 20 视频

找出播放量最高的视频，分析爆款视频的特征。

In [ ]:
# ============================================================
# orderBy desc: 按播放量降序排列，取 Top 20
# ============================================================
top_videos = clean_df.select(
    'title', 'category', 'creator_name', 'pub_date',
    'view', 'like', 'coin', 'favorite', 'share', 'duration_min'
).orderBy(F.desc('view')).limit(20)

print('=== 播放量 Top 20 视频 ===')
top_videos.show(truncate=35)

### 4.3 UP主排行榜分析

统计各 UP主的上榜次数和总播放量。

In [ ]:
# ============================================================
# 按 UP主聚合：上榜次数、总播放量、平均播放量
# ============================================================
creator_rank = clean_df.groupBy('creator_mid', 'creator_name').agg(
    F.count('*').alias('appear_count'),          # 上榜次数
    F.sum('view').alias('total_view'),           # 累计播放量
    F.round(F.avg('view'), 0).alias('avg_view'), # 平均播放量
    F.sum('like').alias('total_like'),           # 累计点赞
).orderBy(F.desc('appear_count'))

print('=== 上榜次数 Top 15 UP主 ===')
creator_rank.show(15, truncate=30)

print('=== 累计播放量 Top 15 UP主 ===')
creator_rank.orderBy(F.desc('total_view')).show(15, truncate=30)

### 4.4 互动指标相关性分析

计算播放量与各互动指标之间的相关系数，分析哪些指标与播放量关系最密切。

In [ ]:
# ============================================================
# corr: 计算两个列之间的 Pearson 相关系数
# 相关系数范围 [-1, 1]，越接近 1 表示正相关越强
# ============================================================
metrics = ['like', 'coin', 'favorite', 'share', 'danmaku', 'reply']

print('=== 播放量与各指标的相关系数 ===')
for metric in metrics:
    corr_val = clean_df.stat.corr('view', metric)
    bar = '█' * int(abs(corr_val) * 30)
    print(f'  view vs {metric:>10s}: {corr_val:.4f}  {bar}')

# ============================================================
# 进一步：计算各互动指标之间的相关性矩阵
# ============================================================
print('\n=== 互动指标间的相关系数矩阵 ===')
all_metrics = ['view'] + metrics
header = f'{"":>12s}' + ''.join(f'{m:>12s}' for m in all_metrics)
print(header)
for m1 in all_metrics:
    row = f'{m1:>12s}'
    for m2 in all_metrics:
        c = clean_df.stat.corr(m1, m2)
        row += f'{c:>12.4f}'
    print(row)

### 4.5 视频时长分布与播放量关系

将视频按时长分段，分析不同时长区间的播放表现。

In [ ]:
# ============================================================
# when/otherwise: Spark 的条件表达式，类似 SQL 的 CASE WHEN
# 将视频按时长划分为多个区间
# ============================================================
duration_df = clean_df.withColumn(
    'duration_group',
    F.when(F.col('duration_min') <= 1, '0-1 min')
     .when(F.col('duration_min') <= 3, '1-3 min')
     .when(F.col('duration_min') <= 5, '3-5 min')
     .when(F.col('duration_min') <= 10, '5-10 min')
     .when(F.col('duration_min') <= 20, '10-20 min')
     .when(F.col('duration_min') <= 60, '20-60 min')
     .otherwise('60+ min')
)

# 按时长区间聚合统计
duration_stats = duration_df.groupBy('duration_group').agg(
    F.count('*').alias('count'),
    F.round(F.avg('view'), 0).alias('avg_view'),
    F.round(F.avg('like'), 0).alias('avg_like'),
    F.round(F.avg('coin'), 0).alias('avg_coin'),
).orderBy('duration_group')

print('=== 不同时长区间的视频统计 ===')
duration_stats.show(truncate=False)

### 4.6 年度趋势分析

分析不同年份的视频数量、平均播放量变化趋势。

In [ ]:
# ============================================================
# 按年份聚合，观察"每周必看"的内容变化趋势
# ============================================================
yearly_trend = clean_df.groupBy('pub_year').agg(
    F.count('*').alias('video_count'),
    F.round(F.avg('view'), 0).alias('avg_view'),
    F.round(F.avg('like'), 0).alias('avg_like'),
    F.round(F.avg('duration_min'), 1).alias('avg_duration'),
    F.countDistinct('creator_mid').alias('unique_creators'),
).orderBy('pub_year')

print('=== 年度趋势统计 ===')
yearly_trend.show(truncate=False)

### 4.7 投币率与收藏率分析

投币率（coin/view）和收藏率（favorite/view）反映了用户对视频质量的认可程度。

In [ ]:
# ============================================================
# 计算各分区的平均投币率和收藏率
# 使用 Spark SQL 函数计算比率
# ============================================================
engagement_df = clean_df.withColumn(
    'coin_rate', F.col('coin') / F.col('view')
).withColumn(
    'fav_rate', F.col('favorite') / F.col('view')
).withColumn(
    'like_rate', F.col('like') / F.col('view')
)

engagement_by_cat = engagement_df.groupBy('category').agg(
    F.count('*').alias('count'),
    F.round(F.avg('coin_rate') * 100, 3).alias('avg_coin_rate_pct'),
    F.round(F.avg('fav_rate') * 100, 3).alias('avg_fav_rate_pct'),
    F.round(F.avg('like_rate') * 100, 3).alias('avg_like_rate_pct'),
).filter(F.col('count') >= 50) \
 .orderBy(F.desc('avg_coin_rate_pct'))

print('=== 各分区互动率排行（按投币率，仅展示 >= 50 条视频的分区）===')
engagement_by_cat.show(15, truncate=False)

## 五、数据可视化

使用 matplotlib 和 seaborn 对 Spark 分析结果进行可视化展示。

In [ ]:
import matplotlib.pyplot as plt
import matplotlib
import seaborn as sns
import pandas as pd

# 设置中文字体，防止图表中文乱码
matplotlib.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'Arial Unicode MS']
matplotlib.rcParams['axes.unicode_minus'] = False

# 设置图表风格
sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 120

print('可视化库加载完成')

### 5.1 分区视频数量分布（柱状图）

In [ ]:
# ============================================================
# 将 Spark DataFrame 转为 Pandas 后绘图
# toPandas(): 将分布式数据收集到 Driver 端
# ============================================================
cat_pdf = category_stats.limit(15).toPandas()

fig, ax = plt.subplots(figsize=(12, 6))
bars = ax.barh(cat_pdf['category'][::-1], cat_pdf['video_count'][::-1],
               color='skyblue', edgecolor='black', linewidth=0.5)

# 在柱子右侧标注数值
for bar, val in zip(bars, cat_pdf['video_count'][::-1]):
    ax.text(bar.get_width() + 5, bar.get_y() + bar.get_height()/2,
            str(val), va='center', fontsize=10)

ax.set_xlabel('视频数量', fontsize=12)
ax.set_title('B站每周必看 Top 15 分区视频数量分布', fontsize=14, fontweight='bold')
ax.set_xlim(0, cat_pdf['video_count'].max() * 1.15)
plt.tight_layout()
plt.savefig('figures/01_category_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('图 1: 分区视频数量分布')

### 5.2 播放量 Top 15 视频（柱状图）

In [ ]:
# ============================================================
# Top 15 播放量视频可视化
# ============================================================
top15_pdf = clean_df.select('title', 'creator_name', 'view', 'category') \
    .orderBy(F.desc('view')).limit(15).toPandas()

# 截断过长的标题
top15_pdf['short_title'] = top15_pdf['title'].apply(
    lambda x: (x[:18] + '...') if len(str(x)) > 18 else str(x)
)

fig, ax = plt.subplots(figsize=(12, 7))
colors = plt.cm.viridis([i / 15 for i in range(15)])
bars = ax.barh(top15_pdf['short_title'][::-1],
               top15_pdf['view'][::-1] / 1e6,
               color=colors, edgecolor='black', linewidth=0.5)

for bar, val in zip(bars, top15_pdf['view'][::-1]):
    ax.text(bar.get_width() + 0.05, bar.get_y() + bar.get_height()/2,
            f'{val/1e4:.0f}万', va='center', fontsize=9)

ax.set_xlabel('播放量（百万）', fontsize=12)
ax.set_title('B站每周必看 播放量 Top 15 视频', fontsize=14, fontweight='bold')
ax.set_xlim(0, top15_pdf['view'].max() / 1e6 * 1.2)
plt.tight_layout()
plt.savefig('figures/02_top15_videos.png', dpi=150, bbox_inches='tight')
plt.show()
print('图 2: 播放量 Top 15 视频')

### 5.3 UP主上榜次数 Top 15

In [ ]:
# ============================================================
# UP主上榜次数排行
# ============================================================
creator_pdf = creator_rank.limit(15).toPandas()

fig, ax = plt.subplots(figsize=(11, 6))
bars = ax.bar(range(len(creator_pdf)), creator_pdf['appear_count'],
              color='coral', edgecolor='black', linewidth=0.5)

# X 轴标签设为 UP主名字，旋转 35 度
ax.set_xticks(range(len(creator_pdf)))
ax.set_xticklabels(creator_pdf['creator_name'], rotation=35, ha='right', fontsize=9)

for bar, val in zip(bars, creator_pdf['appear_count']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
            str(val), ha='center', fontsize=10)

ax.set_ylabel('上榜次数', fontsize=12)
ax.set_title('B站每周必看 上榜次数 Top 15 UP主', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('figures/03_top_creators.png', dpi=150, bbox_inches='tight')
plt.show()
print('图 3: UP主上榜次数 Top 15')

### 5.4 互动指标相关性热力图

In [ ]:
# ============================================================
# 使用 Spark 计算相关系数矩阵，再用 seaborn 绘制热力图
# ============================================================
metrics_cn = ['view', 'like', 'coin', 'favorite', 'share', 'danmaku', 'reply']
labels_cn = ['播放量', '点赞', '投币', '收藏', '分享', '弹幕', '评论']

# 构建相关系数矩阵
corr_matrix = []
for m1 in metrics_cn:
    row = []
    for m2 in metrics_cn:
        row.append(clean_df.stat.corr(m1, m2))
    corr_matrix.append(row)

corr_pdf = pd.DataFrame(corr_matrix, index=labels_cn, columns=labels_cn)

# 绘制热力图
fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(corr_pdf, annot=True, fmt='.2f', cmap='RdYlBu_r',
            center=0, vmin=-1, vmax=1, linewidths=0.5, ax=ax,
            square=True, cbar_kws={'shrink': 0.8})
ax.set_title('互动指标相关系数热力图', fontsize=14, fontweight='bold', pad=15)
plt.tight_layout()
plt.savefig('figures/04_correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print('图 4: 互动指标相关性热力图')

### 5.5 视频时长 vs 播放量散点图

In [ ]:
# ============================================================
# 散点图：视频时长 vs 播放量
# 随机采样 2000 条避免过度重叠
# ============================================================
sample_pdf = clean_df.select('duration_min', 'view', 'category') \
    .sample(fraction=0.15, seed=42).toPandas()

fig, ax = plt.subplots(figsize=(12, 6))
ax.scatter(sample_pdf['duration_min'], sample_pdf['view'] / 1e4,
           alpha=0.3, s=15, c='steelblue', edgecolors='none')

ax.set_xlabel('视频时长（分钟）', fontsize=12)
ax.set_ylabel('播放量（万）', fontsize=12)
ax.set_title('视频时长 vs 播放量', fontsize=14, fontweight='bold')
ax.set_xlim(0, min(sample_pdf['duration_min'].quantile(0.99), 120))
ax.set_ylim(0, sample_pdf['view'].quantile(0.99) / 1e4)
plt.tight_layout()
plt.savefig('figures/05_duration_vs_view.png', dpi=150, bbox_inches='tight')
plt.show()
print('图 5: 视频时长 vs 播放量散点图')

### 5.6 年度趋势折线图

In [ ]:
# ============================================================
# 年度趋势：视频数量 & 平均播放量双轴折线图
# ============================================================
yearly_pdf = yearly_trend.toPandas()

fig, ax1 = plt.subplots(figsize=(12, 5))

# 左轴：视频数量
color1 = 'tab:blue'
ax1.set_xlabel('年份', fontsize=12)
ax1.set_ylabel('视频数量', color=color1, fontsize=12)
line1 = ax1.plot(yearly_pdf['pub_year'], yearly_pdf['video_count'],
                 marker='o', color=color1, linewidth=2, label='视频数量')
ax1.tick_params(axis='y', labelcolor=color1)

# 右轴：平均播放量
ax2 = ax1.twinx()
color2 = 'tab:red'
ax2.set_ylabel('平均播放量（万）', color=color2, fontsize=12)
line2 = ax2.plot(yearly_pdf['pub_year'], yearly_pdf['avg_view'] / 1e4,
                 marker='s', color=color2, linewidth=2, label='平均播放量')
ax2.tick_params(axis='y', labelcolor=color2)

ax1.set_title('B站每周必看 年度趋势', fontsize=14, fontweight='bold')
ax1.set_xticks(yearly_pdf['pub_year'])
lines = line1 + line2
ax1.legend(lines, [l.get_label() for l in lines], loc='upper left')
plt.tight_layout()
plt.savefig('figures/06_yearly_trend.png', dpi=150, bbox_inches='tight')
plt.show()
print('图 6: 年度趋势折线图')

### 5.7 各分区平均互动率对比

In [ ]:
# ============================================================
# 各分区投币率、收藏率、点赞率对比（分组柱状图）
# ============================================================
eng_pdf = engagement_by_cat.limit(12).toPandas()

x = range(len(eng_pdf))
width = 0.25

fig, ax = plt.subplots(figsize=(14, 6))
ax.bar([i - width for i in x], eng_pdf['avg_coin_rate_pct'],
       width, label='投币率', color='#FF6B6B', edgecolor='black', linewidth=0.3)
ax.bar(x, eng_pdf['avg_fav_rate_pct'],
       width, label='收藏率', color='#4ECDC4', edgecolor='black', linewidth=0.3)
ax.bar([i + width for i in x], eng_pdf['avg_like_rate_pct'],
       width, label='点赞率', color='#45B7D1', edgecolor='black', linewidth=0.3)

ax.set_xticks(x)
ax.set_xticklabels(eng_pdf['category'], rotation=30, ha='right', fontsize=9)
ax.set_ylabel('比率 (%)', fontsize=12)
ax.set_title('各分区互动率对比（投币率 / 收藏率 / 点赞率）', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
plt.tight_layout()
plt.savefig('figures/07_engagement_by_category.png', dpi=150, bbox_inches='tight')
plt.show()
print('图 7: 各分区互动率对比')

### 5.8 视频时长区间分布饼图

In [ ]:
# ============================================================
# 视频时长区间占比饼图
# ============================================================
dur_pdf = duration_stats.toPandas()

fig, ax = plt.subplots(figsize=(8, 8))
wedges, texts, autotexts = ax.pie(
    dur_pdf['count'],
    labels=dur_pdf['duration_group'],
    autopct='%1.1f%%',
    colors=sns.color_palette('pastel'),
    startangle=90,
    pctdistance=0.8,
    wedgeprops=dict(edgecolor='white', linewidth=1.5)
)
for t in autotexts:
    t.set_fontsize(10)
    t.set_fontweight('bold')

ax.set_title('视频时长区间分布', fontsize=14, fontweight='bold', pad=20)
plt.tight_layout()
plt.savefig('figures/08_duration_pie.png', dpi=150, bbox_inches='tight')
plt.show()
print('图 8: 视频时长区间分布饼图')

## 六、亮点与总结

### 6.1 亮点

1. **数据规模大**：本项目基于 377 期、约 14000 条真实 B站"每周必看"视频数据进行分析，数据量充足。
2. **Spark 全流程**：从 JSON 数据读取、嵌套结构展开、数据清洗到聚合分析，全部使用 PySpark 完成，充分发挥了 Spark 的分布式处理能力。
3. **多维度分析**：从分区分布、UP主排行、互动相关性、时长分布、年度趋势等多个维度对数据进行了深入分析。
4. **图文结合**：每个分析环节都配有可视化图表，并对图表进行了文字描述和分析。

### 6.2 总结

（在此处撰写 200 字以上的心得体会，可以包括：）

- 通过本次项目掌握了 PySpark 的基本使用方法，包括 DataFrame 操作、数据清洗、聚合分析等。
- 学会了使用 `explode` 处理嵌套 JSON 数据，以及使用 `Window` 函数进行去重操作。
- 在数据可视化方面，将 Spark 分析结果转为 Pandas DataFrame 后使用 matplotlib 和 seaborn 绘图，实现了分析与展示的分离。
- 遇到的主要问题包括：JSON 嵌套结构展开的处理、中文乱码的解决、以及大数据量下图表的性能优化。
- 通过本项目加深了对大数据分析流程的理解，提升了数据处理与分析的实践能力。

---

In [ ]:
# ============================================================
# 关闭 SparkSession，断开与 Spark Connect Server 的连接
# 注意: 这只会关闭客户端会话，不会停止 Server 本身
# ============================================================
spark.stop()
print('SparkSession 已关闭，Spark Connect Server 仍在运行')